# Analytics Hub - Access Scope

Org-scoped access (opt-in row-level security). The installer's **access** command manages who sees what and
runs this notebook; with org-scoped access on, the pipeline runs it too, before the models refresh.

**Data** (`MODE = 'all'`, needs the default Lakehouse):

| Table | What it holds |
|---|---|
| `dbo.access_scope` | Who sees what: one row per person and scope (`all`, `hierarchy` below a person, or an `org` value). `*` is the default for anyone without rows. Loaded from `Files/analytics_hub_access/access_scope.csv`, which the installer writes. |
| `dbo.access_people` | Each person's manager chain as lower-case UPNs joined by `|`, top first, from `dbo.copilot_org_data`. |

Rows granted in the app (`GrantedVia = share`) are kept when the installer's file is loaded.

**Role membership** (`ROLES_FILE` or `ROLES_B64`, set by the installer only): re-applies each model's two roles, members
included, through the XMLA endpoint. *Everyone* has no filters except on `Access Scope`; its members are the people
with an `all` scope (and the Viewers group when the default is `all`). *Scoped viewers* filters by org; its member is
the Analytics Hub Viewers group. Power BI combines a person's roles, so someone in both sees everything. Needs a
Fabric or Premium capacity. `MODE = 'membership'` does only this, without a Lakehouse.

Models without org-scoped access never read these tables.

In [ ]:
# === CONFIG ===
MODE         = 'all'                                         # 'all': load the tables (and roles, if given); 'membership': roles only
SCOPE_FILE   = 'Files/analytics_hub_access/access_scope.csv'  # written by the installer's access command
SCOPE_TABLE  = 'dbo.access_scope'
PEOPLE_TABLE = 'dbo.access_people'
ORG_TABLE    = 'dbo.copilot_org_data'
ROLES_B64    = ''                                            # set by the installer: base64 JSON [{"dataset": name, "role": {...}}]
ROLES_FILE   = ''                                            # or the same JSON in a file: Files/... in the Lakehouse, or an abfss:// URL
WORKSPACE_ID = ''                                            # the models' workspace; empty means this one

In [ ]:
# === HELPERS (pure, so they can be tested off Fabric; same rules as valuelens_core.access / org) ===
import base64
import csv
import io
import json

SCOPE_COLUMNS = ['UserPrincipalName', 'ScopeType', 'ScopeValue', 'CanDelegate', 'GrantedBy', 'GrantedVia',
                 'Status', 'Note', 'CreatedAt']
SCOPE_TYPES = ('all', 'hierarchy', 'org')
STATUSES = ('active', 'pending_group')
MANAGED_VIA = ('seed', 'installer')


def parse_scope_csv(text):
    reader = csv.DictReader(io.StringIO((text or '').lstrip('\ufeff')))
    header = [h.strip() for h in (reader.fieldnames or [])]
    missing = [c for c in ('UserPrincipalName', 'ScopeType') if c not in header]
    if missing:
        raise ValueError(f'access_scope.csv is missing column(s): {", ".join(missing)}')
    out, seen = [], set()
    for line, raw in enumerate(reader, start=2):
        row = {(k or '').strip(): (v or '').strip() for k, v in raw.items()}
        upn = row.get('UserPrincipalName', '').lower()
        kind = row.get('ScopeType', '').lower()
        if not upn and not kind:
            continue
        if not upn or (upn != '*' and '@' not in upn):
            raise ValueError(f'access_scope.csv line {line}: UserPrincipalName must be a UPN or *.')
        if kind not in SCOPE_TYPES:
            raise ValueError(f'access_scope.csv line {line}: ScopeType must be one of {", ".join(SCOPE_TYPES)}.')
        value = row.get('ScopeValue', '')
        if kind == 'all':
            value = ''
        elif kind == 'hierarchy':
            value = value.lower()
            if '@' not in value:
                raise ValueError(f'access_scope.csv line {line}: a hierarchy scope needs the root person\'s UPN.')
        elif not value:
            raise ValueError(f'access_scope.csv line {line}: an org scope needs a value.')
        via = row.get('GrantedVia', '').lower() or 'installer'
        if via not in MANAGED_VIA:
            raise ValueError(f'access_scope.csv line {line}: GrantedVia must be seed or installer.')
        status = row.get('Status', '').lower() or 'active'
        if status not in STATUSES:
            raise ValueError(f'access_scope.csv line {line}: Status must be active or pending_group.')
        key = (upn, kind, value)
        if key in seen:
            continue
        seen.add(key)
        out.append({
            'UserPrincipalName': upn,
            'ScopeType': kind,
            'ScopeValue': value,
            'CanDelegate': 'true' if row.get('CanDelegate', '').lower() in ('true', '1', 'yes') else 'false',
            'GrantedBy': row.get('GrantedBy', '').lower(),
            'GrantedVia': via,
            'Status': status,
            'Note': row.get('Note', ''),
            'CreatedAt': row.get('CreatedAt', ''),
        })
    return out


def merge_scope(existing, loaded):
    # The file replaces the installer-managed rows; rows granted in the app stay unless the file has the same key.
    keys = {(r['UserPrincipalName'], r['ScopeType'], r['ScopeValue']) for r in loaded}
    kept = [r for r in existing if (r.get('GrantedVia') or '') not in MANAGED_VIA
            and (r['UserPrincipalName'], r['ScopeType'], r['ScopeValue']) not in keys]
    return kept + list(loaded)


def org_paths(rows):
    # Each person's managerUPN chain, top first; a cycle truncates it (as HierarchyPath does).
    def _n(v):
        return (v or '').strip().lower()

    mgr_of = {}
    for row in rows:
        user = _n(row.get('PersonId'))
        if user:
            mgr_of[user] = _n(row.get('managerUPN'))
    out = {}
    for user in mgr_of:
        chain, seen, cur = [], set(), user
        while cur and cur in mgr_of and cur not in seen:
            seen.add(cur)
            chain.append(cur)
            manager = mgr_of[cur]
            cur = '' if manager == cur else manager
        out[user] = '|'.join(reversed(chain))
    return out


def access_people(rows):
    def _n(v):
        return (v or '').strip().lower()

    computed = None
    out = {}
    for row in rows:
        person = _n(row.get('PersonId'))
        if not person or person in out:
            continue
        path = _n(row.get('OrgPath'))
        if not path:
            if computed is None:
                computed = org_paths(rows)
            path = computed.get(person) or person
        out[person] = path
    return sorted(out.items())


def role_scripts(roles_b64):
    # TMSL createOrReplace for each role, members included: one script per model.
    if not (roles_b64 or '').strip():
        return []
    return role_scripts_json(base64.b64decode(roles_b64).decode('utf-8'))


def role_scripts_json(text):
    if not (text or '').strip():
        return []
    items = json.loads(text)
    return [(i['dataset'], {'createOrReplace': {'object': {'database': i['dataset'], 'role': i['role']['name']},
                                                 'role': i['role']}}) for i in items]

In [ ]:
# === LOAD ===
import os
from datetime import datetime, timezone

NOW = datetime.now(timezone.utc).replace(tzinfo=None, microsecond=0)
SUMMARY = {'mode': MODE}


def _scope_schema():
    from pyspark.sql import types as T
    return T.StructType([
        T.StructField('UserPrincipalName', T.StringType()), T.StructField('ScopeType', T.StringType()),
        T.StructField('ScopeValue', T.StringType()), T.StructField('CanDelegate', T.BooleanType()),
        T.StructField('GrantedBy', T.StringType()), T.StructField('GrantedVia', T.StringType()),
        T.StructField('Status', T.StringType()), T.StructField('Note', T.StringType()),
        T.StructField('CreatedAt', T.TimestampType()),
    ])


def _as_row(r):
    created = r.get('CreatedAt')
    if isinstance(created, str):
        try:
            created = datetime.fromisoformat(created.replace('Z', '+00:00')) if created else None
        except ValueError:
            created = None
    if created is not None and created.tzinfo is not None:
        created = created.astimezone(timezone.utc).replace(tzinfo=None)
    can = r.get('CanDelegate')
    return (r['UserPrincipalName'], r['ScopeType'], r['ScopeValue'] or '',
            can if isinstance(can, bool) else str(can).lower() == 'true',
            r.get('GrantedBy') or None, r.get('GrantedVia') or 'installer', r.get('Status') or 'active',
            r.get('Note') or None, created or NOW)


def load_scope():
    path = f'/lakehouse/default/{SCOPE_FILE}'
    exists = spark.catalog.tableExists(SCOPE_TABLE)
    if not os.path.isfile(path):
        if not exists:
            spark.createDataFrame([], _scope_schema()).write.format('delta').saveAsTable(SCOPE_TABLE)
            print(f'Created an empty {SCOPE_TABLE}: nobody has a scope yet.')
        return None
    with open(path, encoding='utf-8') as fh:
        loaded = parse_scope_csv(fh.read())
    existing = [r.asDict() for r in spark.table(SCOPE_TABLE).collect()] if exists else []
    rows = merge_scope(existing, loaded)
    (spark.createDataFrame([_as_row(r) for r in rows], _scope_schema())
        .write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(SCOPE_TABLE))
    print(f'{SCOPE_TABLE}: {len(loaded)} row(s) from the installer, {len(rows) - len(loaded)} granted in the app.')
    return len(rows)


def load_people():
    from pyspark.sql import types as T
    schema = T.StructType([T.StructField('PersonId', T.StringType()), T.StructField('OrgPath', T.StringType())])

    def keep(why):
        # The models read this table, so it must be there even before org data is.
        if not spark.catalog.tableExists(PEOPLE_TABLE):
            spark.createDataFrame([], schema).write.format('delta').saveAsTable(PEOPLE_TABLE)
            print(f'{why}; created an empty {PEOPLE_TABLE}.')
        else:
            print(f'{why}; {PEOPLE_TABLE} is left as it is.')

    if not spark.catalog.tableExists(ORG_TABLE):
        keep(f'{ORG_TABLE} is not there yet')
        return None
    df = spark.table(ORG_TABLE)
    cols = [c for c in ('PersonId', 'managerUPN', 'OrgPath') if c in df.columns]
    if 'PersonId' not in cols:
        keep(f'{ORG_TABLE} has no PersonId column')
        return None
    people = access_people([r.asDict() for r in df.select(*cols).collect()])
    (spark.createDataFrame(people, schema)
        .write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(PEOPLE_TABLE))
    print(f'{PEOPLE_TABLE}: {len(people)} people.')
    return len(people)


if MODE == 'all':
    SUMMARY['scopeRows'] = load_scope()
    SUMMARY['people'] = load_people()

In [ ]:
# === ROLE MEMBERSHIP ===
import notebookutils


def read_roles_file(path):
    # A Lakehouse path (Files/...) or a storage URL (abfss://...), read as the person running the notebook.
    if not (path or '').strip():
        return ''
    if '://' not in path:
        local = f'/lakehouse/default/{path}'
        if not os.path.isfile(local):
            raise FileNotFoundError(f'{path} is not in the Lakehouse.')
        with open(local, encoding='utf-8') as fh:
            return fh.read()
    local = '/tmp/analytics_hub_access_roles.json'
    notebookutils.fs.cp(path, f'file:{local}')
    with open(local, encoding='utf-8') as fh:
        return fh.read()


SCRIPTS = role_scripts(ROLES_B64) or role_scripts_json(read_roles_file(ROLES_FILE))
if SCRIPTS:
    import sempy.fabric as fabric
    SUMMARY['roles'] = []
    for dataset, script in SCRIPTS:
        fabric.execute_tmsl(json.dumps(script), workspace=WORKSPACE_ID or None)
        role = script['createOrReplace']['object']['role']
        SUMMARY['roles'].append(f'{dataset}: {role}')
        print(f'{dataset}: {role} role and members applied.')
notebookutils.notebook.exit(json.dumps(SUMMARY))